In [1]:
################### Make Timeseries of Constant Forcing ######################
# The purpose of this script is to make timeseries of constant values
# for air temperature, pressure, humidity, precipitation, and cloud cover
# for use in the idealized ROMs-Budgell model. The user only needs to 
# change the grid file the shapes are based on and the values (if desired).
#
# Notes:
# - 
#
###############################################################################

In [2]:
# Load in the packages
import xarray as xr
import numpy as np
import cartopy
import glob
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.feature as cfeature
import cmocean.cm as cmo
import matplotlib.pyplot as plt
import matplotlib.ticker as tick
import warnings
import numpy as np
from netCDF4 import Dataset
import datetime
from datetime import datetime
import matplotlib.dates as mdates
import pandas as pd
import xroms
from matplotlib import ticker
crs = ccrs.PlateCarree()
warnings.filterwarnings("ignore") 
#Cartopy
land_10m = cfeature.NaturalEarthFeature('physical', 'land', '10m',
                                edgecolor='face',
                                facecolor=cfeature.COLORS['land'])

from xhistogram.xarray import histogram

In [3]:
# Make time
# Starting at beginning of year
hourly_datetimes = pd.date_range(start='2000-01-01', periods=366*24, freq='H')
# Make shifted datetimes (using this for now but ultimately probably either would work
# since they are later shifted to have first time step as 0)
hourly_datetimes_shifted = pd.date_range(start='2000-10-15', periods=366*24, freq='D') # freq='H'
#hourly_datetimes_shifted = pd.date_range(start='2000-10-15', end='2001-10-15', freq='D') # freq='H'


In [4]:
len(hourly_datetimes_shifted)

8784

In [5]:
# Set the values for each variable 
air_temp = -25 # celsius
air_pres = 1015.14 # mb
air_rh = 0.87 # percentage
precip = 0.0 # kg/m2s
cloud_cover = 0.87 # fraction
longwave_down = 250 # W/m2
net_shortwave = 0.0 # W/m2

In [6]:
# Load in the grid 
# Load in the idealized model grid to get the dimensions 
# 500 m
#grid = xr.open_dataset('/global/homes/b/bundzis/Projects/Beaufort_ROMS_idealized_jet/Include/grd_500m_002.nc')
grid = xr.open_dataset('/global/homes/b/bundzis/Projects/Beaufort_ROMS_idealized_jet_updated/Include/grd_500m_span_300km_001.nc')
# 1 km
#grid = xr.open_dataset('/global/homes/b/bundzis/Projects/Beaufort_ROMS_idealized_jet/Include/grd_1km_span_500km_noise_001.nc')

In [7]:
grid

<xarray.Dataset> Size: 29MB
Dimensions:      (eta_rho: 602, xi_rho: 402, eta_psi: 601, xi_psi: 401,
                  eta_u: 602, xi_u: 401, eta_v: 601, xi_v: 402)
Dimensions without coordinates: eta_rho, xi_rho, eta_psi, xi_psi, eta_u, xi_u,
                                eta_v, xi_v
Data variables: (12/18)
    x_rho        (eta_rho, xi_rho) float64 2MB ...
    y_rho        (eta_rho, xi_rho) float64 2MB ...
    x_psi        (eta_psi, xi_psi) float64 2MB ...
    y_psi        (eta_psi, xi_psi) float64 2MB ...
    x_u          (eta_u, xi_u) float64 2MB ...
    y_u          (eta_u, xi_u) float64 2MB ...
    ...           ...
    angle        (eta_rho, xi_rho) float64 2MB ...
    spherical    bool 1B ...
    xl           float64 8B ...
    el           float64 8B ...
    visc_factor  (eta_rho, xi_rho) float64 2MB ...
    diff_factor  (eta_rho, xi_rho) float64 2MB ...

In [8]:
# Pull out grid dimensions 
# Read in the dimensions
# rho
eta_rho_len = len(grid.eta_rho) # 206
xi_rho_len = len(grid.xi_rho) # 608
print('eta_rho_len: ', eta_rho_len)
print('xi_rho_len: ', xi_rho_len)
# u
eta_u_len = len(grid.eta_u) # 206
xi_u_len = len(grid.xi_u) # 607
print('eta_u_len: ', eta_u_len)
print('xi_u_len: ', xi_u_len)
# v
eta_v_len = len(grid.eta_v) #
xi_v_len = len(grid.xi_v) # 
print('eta_v_len: ', eta_v_len)
print('xi_v_len: ', xi_v_len)

# Define other dimension lengths
# eta rho
Mp = len(grid.eta_rho)
# xi rho
Lp = len(grid.xi_rho)
print('Mp: ', Mp)
print('Lp: ', Lp)

# # latitude
# lat_u_len = len(grid.lat_u)
# lat_v_len = len(grid.lat_v)
# print('lat_u_len: ', lat_u_len)
# print('lat_v_len: ', lat_v_len)

# # longitude
# lon_u_len = len(grid.lon_u)
# lon_v_len = len(grid.lon_v)
# print('lon_u_len: ', lon_u_len)
# print('lon_v_len: ', lon_v_len)

eta_rho_len:  602
xi_rho_len:  402
eta_u_len:  602
xi_u_len:  401
eta_v_len:  601
xi_v_len:  402
Mp:  602
Lp:  402


In [9]:
# Need to make a time that is hours since some reference time 
# Get the length of time for the data
# Set the number of hourly time steps
num_time = len(hourly_datetimes_shifted)

# convert all the times to seconds since the first time step so it is on generic time)
#time_tmp = ((hourly_datetimes_shifted[:] - datetime(0,12,31)).total_seconds() - 86400)
time_tmp_1 = ((hourly_datetimes_shifted[:] - hourly_datetimes_shifted[0]).total_seconds())
# Trim to just the time we want
time_tmp = time_tmp_1[:num_time]
# Get the length from this 
# time
time_len = len(time_tmp)

In [10]:
print(time_len)

8784


In [11]:
# Make a version of the code that is this shape and same everywhere
const_air_temp_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))
const_air_pres_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))
const_air_rh_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))
const_precip_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))
const_cloud_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))
const_longwave_down_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))
const_net_shortwave_ongrid = np.empty((time_len, eta_rho_len, xi_rho_len))

# Loop through space and fill with the idealized version
for y in range(eta_rho_len):
    for x in range(xi_rho_len):
        const_air_temp_ongrid[:,y,x] = air_temp
        const_air_pres_ongrid[:,y,x] = air_pres
        const_air_rh_ongrid[:,y,x] = air_rh
        const_precip_ongrid[:,y,x] = precip
        const_cloud_ongrid[:,y,x] = cloud_cover
        const_longwave_down_ongrid[:,y,x] = longwave_down
        const_net_shortwave_ongrid[:,y,x] = net_shortwave

In [12]:
# Make the netcdf for the forcing file 

# ------------------------------- Create the netCDF file ---------------------------

#name of file I am writing to
# 500 m
const_frc = '/pscratch/sd/b/bundzis/Beaufort_ROMS_idealized_jet_updated_scratch/Forcing_files/idealized_const_forcing_500m_span_300km_001.nc'   #UPDATE PATH
# 1 km
#const_frc = '/pscratch/sd/b/bundzis/Beaufort_ROMS_idealized_jet_updated_scratch/Forcing_files/const_forcing_1km_span_500km_001.nc'   #UPDATE PATH

#create file to write to
nc1 = Dataset(const_frc, 'w', format='NETCDF4')

#Global attributes
global_defaults = dict(gridname = '*.nc',
                      type = 'ROMS grid shaped ERA5 idealized constant forcing',
                      history = 'Created by Brianna Undzis',
                      Conventions = 'CF',
                      Institution = 'University of Colorado Boulder',
                      date = str(datetime.today()))
    
#create dictionary for model
d = {}
d = global_defaults

for att, value in d.items():
    setattr(nc1, att, value)

# Create dimensions
nc1.createDimension('xi_rho',  xi_rho_len)   # rho
nc1.createDimension('eta_rho', eta_rho_len)
# nc1.createDimension('lon',  era5_lon_len)   # rho
# nc1.createDimension('lat', era5_lat_len)
nc1.createDimension('tair_time', None)
nc1.createDimension('pair_time', None)
nc1.createDimension('qair_time', None)
nc1.createDimension('rain_time', None)
nc1.createDimension('cloud_time', None)
nc1.createDimension('lrf_time', None)
nc1.createDimension('srf_time', None)
nc1.createDimension('one',     1)

# Create variables 
# --------------------
# Coordinate Variables
# --------------------
# # era5 lon
# lon = nc1.createVariable('lon', 'd', ('lon',), zlib=True)
# lon.long_name = 'longitude coordinate of ERA5 data'
# lon.standard_name = 'projection_lon_coordinate'
# lon.units = 'degrees'
# lon_tmp = era5_longwave.longitude
# lon[:] = lon_tmp[:]

# # era5 lat
# lat = nc1.createVariable('lat', 'd', ('lat',), zlib=True)
# lat.long_name = 'latitude coordinate of ERA5 data'
# lat.standard_name = 'projection_lat_coordinate'
# lat.units = 'degrees'
# #lat_tmp = era5_longwave.latitude
# lat_tmp = lat_flip # FLIP
# lat[:] = lat_tmp[:]

# xi rho
xi_rho = nc1.createVariable('xi_rho', 'd', ('xi_rho',), zlib=True)
xi_rho.long_name = 'xi coordinate of RHO-points'
xi_rho.standard_name = 'projection_xi_coordinate'
xi_rho.units = 'meter'
xi_rho_tmp = np.arange(0, Lp)
xi_rho[:] = xi_rho_tmp[:]

# eta rho
eta_rho = nc1.createVariable('eta_rho', 'd', ('eta_rho',), zlib=True)
eta_rho.long_name = 'eta coordinate of RHO-points'
eta_rho.standard_name = 'projection_eta_coordinate'
eta_rho.units = 'meter'
eta_rho_tmp = np.arange(0, Mp)
eta_rho[:] = eta_rho_tmp[:]

# tair_time (in seconds)
tair_time_g = nc1.createVariable('tair_time', None, ('tair_time'), zlib=True)
tair_time_g.long_name = 'seconds since 0001-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
tair_time_g.units = 'second'
tair_time_g.field = 'time, scalar, series'
tair_time_g[:] = time_tmp[:]

# pair_time (in seconds)
pair_time_g = nc1.createVariable('pair_time', None, ('pair_time'), zlib=True)
pair_time_g.long_name = 'seconds since 0001-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
pair_time_g.units = 'second'
pair_time_g.field = 'time, scalar, series'
pair_time_g[:] = time_tmp[:]

# qair_time (in seconds)
qair_time_g = nc1.createVariable('qair_time', None, ('qair_time'), zlib=True)
qair_time_g.long_name = 'seconds since 0001-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
qair_time_g.units = 'second'
qair_time_g.field = 'time, scalar, series'
qair_time_g[:] = time_tmp[:]

# rain_time (in seconds)
rain_time_g = nc1.createVariable('rain_time', None, ('rain_time'), zlib=True)
rain_time_g.long_name = 'seconds since 2000-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
rain_time_g.units = 'second'
rain_time_g.field = 'time, scalar, series'
rain_time_g[:] = time_tmp[:]

# cloud_time (in seconds)
cloud_time_g = nc1.createVariable('cloud_time', None, ('cloud_time'), zlib=True)
cloud_time_g.long_name = 'seconds since 2000-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
cloud_time_g.units = 'second'
cloud_time_g.field = 'time, scalar, series'
cloud_time_g[:] = time_tmp[:]

# lrf_time (in seconds)
lrf_time_g = nc1.createVariable('lrf_time', None, ('lrf_time'), zlib=True)
lrf_time_g.long_name = 'seconds since 0001-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
lrf_time_g.units = 'second'
lrf_time_g.field = 'time, scalar, series'
lrf_time_g[:] = time_tmp[:]

# srf_time (in seconds)
srf_time_g = nc1.createVariable('srf_time', None, ('srf_time'), zlib=True)
srf_time_g.long_name = 'seconds since 0001-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
srf_time_g.units = 'second'
srf_time_g.field = 'time, scalar, series'
srf_time_g[:] = time_tmp[:]

# --------------------
# Atmospheric Variables 
# --------------------

# Tair
tair_interp_g = nc1.createVariable('Tair', 'f8', ('tair_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
tair_interp_g.long_name = 'surface_air_temperature'
tair_interp_g.standard_name = 'surface_air_temperature'
tair_interp_g.units = 'Celsius' 
tair_interp_g.coordinates = 'eta_rho xi_rho' 
tair_interp_g[:,:,:] = const_air_temp_ongrid[:,:,:]

# Pair
pair_interp_g = nc1.createVariable('Pair', 'f8', ('pair_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
pair_interp_g.long_name = 'surface_air_pressure'
pair_interp_g.standard_name = 'surface_air_pressure'
pair_interp_g.units = 'mb' 
pair_interp_g.coordinates = 'eta_rho xi_rho' 
pair_interp_g[:,:,:] = const_air_pres_ongrid[:,:,:]

# Qair
qair_interp_g = nc1.createVariable('Qair', 'f8', ('qair_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
qair_interp_g.long_name = 'surface sir relative humidity'
qair_interp_g.standard_name = 'surface_air_relative_humidity'
qair_interp_g.units = 'percentage' 
qair_interp_g.coordinates = 'eta_rho xi_rho' 
qair_interp_g[:,:,:] = const_air_rh_ongrid[:,:,:]

# rain
rain_interp_g = nc1.createVariable('rain', 'f8', ('rain_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
rain_interp_g.long_name = 'rain fall rate'
rain_interp_g.standard_name = 'precipitation_flux'
rain_interp_g.units = 'kiloegram meter-2 second-1'
rain_interp_g.coordinates = 'eta_rho xi_rho' 
rain_interp_g[:,:,:] = const_precip_ongrid[:,:,:]

# cloud cover
cloud_interp_g = nc1.createVariable('cloud', 'f8', ('cloud_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
cloud_interp_g.long_name = 'cloud fraction'
cloud_interp_g.standard_name = 'cloud_area_fraction'
cloud_interp_g.units = 'nondimensional' 
cloud_interp_g.coordinates = 'eta_rho xi_rho'
cloud_interp_g[:,:,:] = const_cloud_ongrid[:,:,:]

# downward longwave radiation flux
lwrad_down_interp_g = nc1.createVariable('lwrad_down', 'f8', ('lrf_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
lwrad_down_interp_g.long_name = 'downward longwave radiation flux'
lwrad_down_interp_g.standard_name = 'surface_downward_longwave_flux'
lwrad_down_interp_g.units = 'watt meter-2' 
lwrad_down_interp_g.coordinates = 'eta_rho xi_rho'
lwrad_down_interp_g[:,:,:] = const_longwave_down_ongrid[:,:,:]

# downward shortwave radiation flux
swrad_interp_g = nc1.createVariable('swrad', 'f8', ('srf_time', 'eta_rho', 'xi_rho'), zlib=True, fill_value=1e30)
swrad_interp_g.long_name = 'net solar shortwave radiation flux'
swrad_interp_g.standard_name = 'net_downward_shortwave_flux_at_sea_water_surface'
swrad_interp_g.units = 'watt meter-2' 
swrad_interp_g.coordinates = 'eta_rho xi_rho'
swrad_interp_g[:,:,:] = const_net_shortwave_ongrid[:,:,:]


nc1.close()
# ------------------------------- End netCDF ---------------------------